In [5]:
from scripts.utils import check_data, preprocess_data, save_data
from scripts.utils import PROJECT_PATH, DATA_PATH, RESULTS_PATH, CELLTYPE_MAP, HISTOTYPE_MAP, ANNDATA_MAP, SUBSETS_CONFIG

from scripts.gene_subsampling import neyman_subsample, thinning
from scripts.studies import study_sparsity, study_sparsity_with_trajectories, study_complete_sparsity, study_group_sparsity, study_group_sparsity_exclude
import numpy as np

from scripts.clustering import cluster_data, find_best_resolution, find_best_resolution_linspace, best_leiden_run

from scripts.utils import plot_UMAP, update_data
import os

import scipy.sparse as sp

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
celltype="Fibroblasts"

fibroblast_cells = check_data(celltype=celltype,data_path=DATA_PATH)
if fibroblast_cells is None:
    fibroblast_cells = preprocess_data(celltype=celltype, with_subsets_config=True, n_neighbors=15, n_comps=100, random_state=42, verbose=False)
    save_data(data=fibroblast_cells, data_path=f"{DATA_PATH}\\{ANNDATA_MAP[celltype]}")

In [11]:
ratio =0.27
thinned_endothelialcells = thinning(fibroblast_cells,reduction_ratio=ratio,same_reads=False,copy=True)

n = thinned_endothelialcells.raw.shape[0]
n_genes_per_cell = ((np.floor(sp.csr_matrix.expm1(thinned_endothelialcells.raw.X).data).astype(int) > 0)).sum(axis=0)


print(n_genes_per_cell / n)

626.1003134796239


In [ ]:
n_cells=2e3
fibroblast_cells =neyman_subsample(data=fibroblast_cells,target_labels=["Fibroblasts"],label_col="celltype_label",stratify_by=["cellstates_tme"],n_target=n_cells)
fibroblast_cells = update_data(data=fibroblast_cells)

In [ ]:
plot_UMAP(fibroblast_cells)

In [ ]:
n_points_ratio = 20
n_runs = 75
n_neighbors_candidates = [15,25,50,100,200,300]

ratio_candidates = np.linspace(0.01,1,n_points_ratio)
labels = fibroblast_cells.obs["cellstates_tme"]
show = True

sparsity_results = study_complete_sparsity(fibroblast_cells,labels=labels,ratio_candidates=ratio_candidates,n_runs=n_runs, n_neighbors_candidates=n_neighbors_candidates, show=show)

In [ ]:
n_points_ratio = 20
n_runs = 75
n_neighbors_candidates = [15,25,50,100,200,300]

ratio_candidates = np.linspace(0.01,0.25,n_points_ratio)
labels = fibroblast_cells.obs["cellstates_tme"]
show = True

sparsity_results = study_complete_sparsity(fibroblast_cells,labels=labels,ratio_candidates=ratio_candidates,n_runs=n_runs, n_neighbors_candidates=n_neighbors_candidates, show=show)

In [ ]:
n_points_ratio = 20
n_runs = 75
ratio_candidates = np.linspace(0.01,1,n_points_ratio)
n_neighbors=20
study_group_sparsity(fibroblast_cells,labels=labels,ratio_candidates=ratio_candidates,n_runs=n_runs, n_neighbors=n_neighbors, search_resolution_method="optuna", show=show)

In [ ]:
n_points_ratio = 20
n_runs = 75
ratio_candidates = np.linspace(0.01,1,n_points_ratio)
n_neighbors=20
study_group_sparsity_exclude(fibroblast_cells,labels=labels,ratio_candidates=ratio_candidates,n_runs=n_runs, n_neighbors=n_neighbors, search_resolution_method="optuna", show=show)

In [ ]:
n_points_ratio = 20
n_runs = 75
ratio_candidates = np.linspace(0.01,0.25,n_points_ratio)
n_neighbors=20
study_group_sparsity_exclude(fibroblast_cells,labels=labels,ratio_candidates=ratio_candidates,n_runs=n_runs, n_neighbors=n_neighbors, search_resolution_method="optuna", show=show)